In [91]:
from dotenv import load_dotenv
from openai import OpenAI
import os
load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.deepseek.com"
)

In [92]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [93]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [19]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally, follow these steps from the FAQ:

1. **Install Ollama** by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:
   - **macOS**: Download the `.pkg` and install it.
   - **Windows**: Download the `.msi` and install it.
   - **Linux**: Run this in the terminal:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **After installation**, open a terminal and type:
   ```bash
   ollama run llama3
   ```
   This command will:
   - Download the LLaMA 3 model (~4GB).
   - Start the model locally.
   - Open a chat-like interface where you can type questions.

3. **To test the Ollama local server**, run:
   ```bash
   curl http://localhost:11434
   ```
   You should receive a response similar to:
   ```json
   {"models": [...]}
   ```

4. **Install the Python client** with:
   ```bash
   pip install ollama
   ```

   Minimal Python example:
   ```python
   import ollama

   response = ollama.chat(
 

In [94]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [95]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [ ]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

# No RAG search database available
response = openai_client.responses.create(
    model='deepseek-api',
    input=messages,
    
)

In [151]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

# Tell the LLM that a search tool exists 
response = openai_client.responses.create(
    model='deepseek-v4-pro',
    # reasoning={"effort": "none"},  # SWITCH OFF THINKING MODE, otherwise it goes to reasoning mode
    input=messages,
    tools=[search_tool] 
)

In [ ]:
# print(openai_client.models.list())


SyncPage[Model](data=[Model(id='deepseek-flash', created=None, object='model', owned_by='deepseek', shutdown_date=None), Model(id='deepseek-v4-pro', created=None, object='model', owned_by='deepseek', shutdown_date=None)], object='list')


In [152]:
call = response.output[1] # response.output[0] in actual nb


In [153]:
call

ResponseFunctionToolCall(arguments='{"query": "Can I join the course?"}', call_id='call_00_mvqiCjVUJVOLabXuwvJr4907', name='search', type='function_call', id='605abf62-b78f-46f4-a156-042a8b282272', async_=None, caller=None, namespace=None, status='completed')

In [154]:
import json
args = json.loads(call.arguments)

In [155]:
call.name

'search'

In [156]:
results = search(**args)

In [157]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not state a total number of hours.'},
 {'id': 'aa310de435',
  'course': 'llm-zoomcamp',
  'section': 'Module 1: RAG',
  'question': 'Can I run the course locally instead of Codespaces?',
  'answer': 'Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.\n\nIf you run locally, make sure you document your set

In [158]:
# 1. Make a call to LLM # We pay
# 2. LLM decided to invoke search(params) -> AGENT
# 3. We have results from the search
# 4. Send the results back to LLM (another call) -> AGENT
# 5. LLM processes the results # We pay again
# 6. LLM gives the answer 

In [159]:
result_json = json.dumps(results, indent=2)

In [160]:
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "5cc511f85b",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Does the course certificate show the number of course hours?",
    "answer": "No. The certificate does not state a total number of hours."
  },
  {
    "id": "aa310de435",
    "course": "llm-zoomcamp",
    "section": "Module 1: RAG",
    "question": "Can I run the course locally instead of Codespaces?",
    "answer": "Yes. Codespaces is just the easiest way for everyone to start with the same environment.\n\nYou can run the course locally if you are comfortable setting up Python, `uv`, Jupyter, Docker, and any other tools needed for the module.

In [161]:
# 4. Send the results back to LLM (another call) -> AGENT
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [162]:
messages.append(call)

In [163]:
messages.append(function_call_output)

In [164]:
messages # Now contains 1. query 2. decision 3. output 

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query": "Can I join the course?"}', call_id='call_00_mvqiCjVUJVOLabXuwvJr4907', name='search', type='function_call', id='605abf62-b78f-46f4-a156-042a8b282272', async_=None, caller=None, namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_00_mvqiCjVUJVOLabXuwvJr4907',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "5cc511f85b",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Does the course certificate show the number of course hours?",\n    "answer": "No. The certificate do

In [165]:
# 5. LLM processes the results and # 6. LLM gives the answer 
response = openai_client.responses.create(
    model='deepseek-flash',
    input=messages, # 
    reasoning={"effort": "none"},
    tools=[search_tool]
)
# This might also not work and it might do another search

In [166]:
print(response.output_text)

Yes, you can still join! Here's what you need to know:

- **Joining is open** — you can just start learning and submitting homework while the forms are open. There's no strict registration check.
- **Certificate caveat**: If you want to receive a certificate, you need to submit your capstone project (and complete the required peer reviews) while a live cohort is still accepting submissions. Homework is not required for the certificate.
- **Self-paced limitation**: You can work through all the material in self-paced mode and prepare your project, but to get a certificate you must submit the project and do peer reviews during a live cohort period.

So feel free to dive into the material right away — and if a certificate matters to you, keep an eye out for the submission window.


In [110]:
# TWO TOTAL REQUESTS

# just for the second call
usage = response.usage
usage.input_tokens, usage.output_tokens

(1048, 140)

In [ ]:
# What if LLM decides to make another tool call? -> One more API request -> Process and give the answer
# -> Continue until no more tool calls required

In [167]:
# Helper: Convert json to python dictionary 

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)
    # elif call.name == 'some other function':
    #   do some other action
    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [168]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [175]:
response = openai_client.responses.create(
    model='deepseek-v4-pro',
    input=messages,
    reasoning = {"effort":"low"},
    tools=[search_tool]
)

In [176]:
response

Response(id='b8f60dd3-f1e2-49af-b0ea-8991d9d4ce6c', created_at=1789231100.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='deepseek-v4-pro', object='response', output=[ResponseOutputMessage(id='37b955ff-5e6f-4e32-9e3f-c7e904730ce2', content=[ResponseOutputText(annotations=[], text='Yes, you can still join! 🎉\n\nThe short answer from the course FAQ is:\n\n> **Yes**, but if you want to receive a certificate, you need to submit your project while we\'re still accepting submissions.\n\nA couple of useful details to keep in mind:\n\n- **You can start learning right away.** You don\'t need a confirmation email or an "acceptance" step — you\'re accepted. Registration is mainly used to gauge interest before the course starts, and it isn\'t checked against any list.\n- **For the certificate**, you need to finish the course with a "live" cohort. That means completing a capstone project and the required peer reviews while submissions are open. Homework is *not* requi

In [177]:
print(response.output_text)

Yes, you can still join! 🎉

The short answer from the course FAQ is:

> **Yes**, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

A couple of useful details to keep in mind:

- **You can start learning right away.** You don't need a confirmation email or an "acceptance" step — you're accepted. Registration is mainly used to gauge interest before the course starts, and it isn't checked against any list.
- **For the certificate**, you need to finish the course with a "live" cohort. That means completing a capstone project and the required peer reviews while submissions are open. Homework is *not* required for the certificate, but the project submission and peer review must happen during the live cohort window.
- **Self-paced learning** is fine for working through the material and preparing your project, but the actual project submission and peer review must occur while a live cohort is accepting them.

So in summary: join

In [178]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

ASSISTANT:
Yes, you can still join! 🎉

The short answer from the course FAQ is:

> **Yes**, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

A couple of useful details to keep in mind:

- **You can start learning right away.** You don't need a confirmation email or an "acceptance" step — you're accepted. Registration is mainly used to gauge interest before the course starts, and it isn't checked against any list.
- **For the certificate**, you need to finish the course with a "live" cohort. That means completing a capstone project and the required peer reviews while submissions are open. Homework is *not* required for the certificate, but the project submission and peer review must happen during the live cohort window.
- **Self-paced learning** is fine for working through the material and preparing your project, but the actual project submission and peer review must occur while a live cohort is accepting them.

So in su

In [ ]:
# Thus, we can also ask a follow-up question

In [179]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='deepseek-flash',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
I'll look up information about joining the course.
function_call: search {"query": "join the course late enrollment"}
function_call: search {"query": "Can I still join the course after it started"}
iteration #2...
ASSISTANT:
Yes, you can still join! Here's what you need to know:

**Joining the course**
- You can start whenever you want — the videos and GitHub materials are available, and there's no strict need to register. As long as the homework submission forms are open, you can submit homework even without registering. Registration is mainly to gauge interest before the start date.

**Getting a certificate**
- If you want to receive a certificate, you need to **submit your Capstone project while submissions are still being accepted**.
- Homework is **not mandatory** for the certificate — only passing the Capstone project is required. Homework points just count toward your leaderboard rank (and it's recommended for reinforcing concepts).

**Where to start**

In [182]:
def agent_loop(instructions, question, model="deepseek-flash") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [183]:
# Typo
agent_loop(instructions, "How do I run Olama locally?")


iteration #1...
ASSISTANT:
I'll look that up for you.
function_call: search {"query": "Ollama run locally"}
function_call: search {"query": "Olama install local model"}
iteration #2...
ASSISTANT:
Let me search for a few related details as well.
function_call: search {"query": "ollama serve localhost port 11434 connection refused"}
function_call: search {"query": "run local model on my machine without API key"}
iteration #3...
ASSISTANT:
It sounds like you mean **Ollama** — a tool that lets you download and run open models (like Llama 3) directly on your own machine, with no API key or paid service. Here's how to do it.

## 1. Install Ollama

Go to [https://ollama.com/download](https://ollama.com/download) and pick your OS:

- **macOS**: download and install the `.pkg`
- **Windows**: download and install the `.msi`
- **Linux**: run this in the terminal:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

## 2. Download and run a model

Open a terminal and run:

```bash
olla

'It sounds like you mean **Ollama** — a tool that lets you download and run open models (like Llama 3) directly on your own machine, with no API key or paid service. Here\'s how to do it.\n\n## 1. Install Ollama\n\nGo to [https://ollama.com/download](https://ollama.com/download) and pick your OS:\n\n- **macOS**: download and install the `.pkg`\n- **Windows**: download and install the `.msi`\n- **Linux**: run this in the terminal:\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\n## 2. Download and run a model\n\nOpen a terminal and run:\n\n```bash\nollama run llama3\n```\n\nThis will:\n- download the LLaMA 3 model (~4 GB),\n- start it locally,\n- drop you into a chat-like prompt where you can type questions.\n\n## 3. Verify the local server\n\nOllama runs a local server on port `11434`. Test it with:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should get a response like:\n\n```json\n{"models": [...]}\n```\n\n## 4. Use it from Python\n\nInstall the client:\n\

In [ ]:
# agent_loop(instructions, "I just discovered the course. Can I still join it?")


In [184]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")


iteration #1...
ASSISTANT:
I'll look up information about joining the course.
function_call: search {"query": "just discovered the course can I join it"}
function_call: search {"query": "join course late enrollment"}
iteration #2...
ASSISTANT:
Let me gather a bit more detail on starting and certification.
function_call: search {"query": "do I need to register to start learning homework"}
function_call: search {"query": "certificate requirements capstone project peer review deadlines"}
iteration #3...
ASSISTANT:
Good news — **yes, you can still join!** 🎉

Here's what you need to know:

**Joining & starting**
- You can start whenever you want — the videos and GitHub materials are all available, and deadlines are listed on the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).
- You don't need to wait for a confirmation email. You're accepted, and you can just start learning and submitting homework while the form is open. Registration is only used to gauge in

'Good news — **yes, you can still join!** 🎉\n\nHere\'s what you need to know:\n\n**Joining & starting**\n- You can start whenever you want — the videos and GitHub materials are all available, and deadlines are listed on the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n- You don\'t need to wait for a confirmation email. You\'re accepted, and you can just start learning and submitting homework while the form is open. Registration is only used to gauge interest before the start date, so it isn\'t checked against any list.\n- A typical weekly workflow is: watch the lesson videos → work through the notebooks/code → read the homework instructions on GitHub → submit answers on the course platform before the deadline.\n\n**About the certificate**\n- If you want to receive a certificate, the key catch is that you must **submit your capstone project while submissions are still being accepted**.\n- Certificates are only available if you finish with a "live" co

Right now the agent will answer anything. Ask it about chess and it will still try. We want a course assistant, not a general chatbot. We tighten the instructions so the agent only answers from the FAQ.

In [185]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query": "queen's gambit"}
function_call: search {"query": "queen gambit chess"}
iteration #2...
function_call: search {"query": "gambit opening chess strategy"}
function_call: search {"query": "course topic queen gambit"}
iteration #3...
ASSISTANT:
I looked this up in the course FAQ, but I couldn't find anything about "Queen's Gambit" there — it's not related to the course or its logistics, so it's outside what I can help with. (For what it's worth, that's a chess opening / a TV series, neither of which is covered in the course materials.)

I can only answer questions based on the course FAQ — things like the LLM Zoomcamp lessons, homework, the capstone project, certificate rules, and course logistics.

Is there anything else about the course you'd like to explore?


'I looked this up in the course FAQ, but I couldn\'t find anything about "Queen\'s Gambit" there — it\'s not related to the course or its logistics, so it\'s outside what I can help with. (For what it\'s worth, that\'s a chess opening / a TV series, neither of which is covered in the course materials.)\n\nI can only answer questions based on the course FAQ — things like the LLM Zoomcamp lessons, homework, the capstone project, certificate rules, and course logistics.\n\nIs there anything else about the course you\'d like to explore?'

This is a lightweight form of an input guardrail. We tell the agent what's in scope and what isn't. A real guardrail checks the input before the agent runs and can block off-topic questions outright. That's a separate topic, but instructions are the first place to start.



Every time you build a new agent, you'd write the same while-loop, the same function-call handling, the same message management.

ToyAIKit wraps this pattern so you can focus on tools, prompts, and behavior.

In [186]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [ ]:
# Here the search_tool is the description or schema - but it is annoying to write 
# this for every tool
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)


In [190]:
# If we add a type hint - query: str and a docstring - """... """ to search, 
# ToyAIKit reads them and derives the schema for us:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [192]:
agent_tools = Tools()
agent_tools.add_tool(search)
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [204]:
# Define the agent 

chat_interface = IPythonChatInterface() # Displays the output 
callback = DisplayingRunnerCallback(chat_interface) # renders model messages and tool calls as they happen. 
# The runner runs the agent loop, the same while True we wrote by hand. 
# It sends messages, executes function calls, adds tool outputs back, and
# repeats until the model is done.
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="deepseek-v4-pro", client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://api.deepseek.com"
))
)


In [205]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


In [206]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


In [ ]:
# Interactive runner - For a chat-like workflow, run the built-in input loop:
# Type questions and get answers. Type "stop" to exit.
runner.run()


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


-> Response received


KeyboardInterrupt: Interrupted by user

Other Frameworks

1. OpenAI Agents SDK
2. PydanticAI - *****
3. LangChain / LangGraph - ****
4. Google ADK


Other frameworks worth knowing:

CrewAI - multi-agent orchestration
AutoGen - multi-agent conversations from Microsoft
Semantic Kernel - from Microsoft, supports C# and Python
Smolagents - lightweight agent framework from HuggingFace
Anthropic Tool Use - Anthropic's native tool use API

Avoiding agents when you can
One closing thought. You don't always need an agent, and that's fine. 
You might assume your app needs one, but often it doesn't.